# Strands Travel Planner (Assignment 2)

An AI agent built with [Strands Agents](https://github.com/strands-agents/sdk-python)
that plans a one-day trip for a city: checks current weather, finds 3 popular
attractions, estimates total visit cost, and writes a one-day itinerary.

## How the workflow fits together

```
┌────────────────────────────────────────────────────────────────┐
│  CITY = "Kathmandu"                                            │
│  PROMPT: "Plan a one-day trip to Kathmandu. Follow these       │
│  steps in order and actually call the tools: ..."              │
└──────────────────────────────┬─────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────┐
│  Strands Agent (OpenAI-compatible model via OpenRouter)        │
│  with 3 tools attached:                                        │
│                                                                │
│  1. get_weather      → OpenWeather current weather (custom)    │
│  2. tavily           → web search for 3 attractions (built-in) │
│  3. calculator       → totals the cost estimate  (built-in)    │
│                                                                │
│  Strands runs the agent loop for us: the model proposes tool   │
│  calls, Strands executes them and feeds results back,          │
│  repeating until the model writes the final itinerary.         │
└──────────────────────────────┬─────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────┐
│  One-day itinerary: weather line, morning/afternoon/evening    │
│  schedule naming the 3 attractions, and a cost breakdown       │
│  with the final total.                                         │
└────────────────────────────────────────────────────────────────┘
```

Unlike assignment 1 (manual function calling, one call visible at a time),
here we use Strands' **automatic agent loop** — the SDK handles the
tool-call round-trips. The prompt still forces an explicit step order
(weather → search → calculate → write), so the workflow stays predictable.

## What you'll see when you run it

The final agent reply looks roughly like:

```
One-Day Trip to Kathmandu

Weather: Current weather in Kathmandu: 22.4C (feels like 22.1C),
scattered clouds.

Morning: Kathmandu Durbar Square — historic palace complex...
Afternoon: Swayambhunath Stupa (Monkey Temple) — hilltop stupa...
Evening: Boudhanath Stupa — evening kora with butter lamps...

Cost breakdown (per person, USD):
- Entry fees: $12
- Local transport: $8
- Meals: $15
Total: $35
```

While it runs, Strands logs each tool call to stdout (tool name, args,
result), so you can watch the loop happen.


## 1. Setup

Install dependencies (from the repo root's shared venv, or locally):

```bash
pip install strands-agents strands-agents-tools requests
```

Create a `.env` file in this folder (copy `.env.example`) with:

- `OPENWEATHER_API_KEY` — openweathermap.org
- `LLM_API_KEY` / `LLM_BASE_URL` / `LLM_MODEL` — OpenAI-compatible endpoint
  (any OpenRouter model works)
- `TAVILY_API_KEY` — tavily.com (the web-search tool reads this itself)


In [ ]:
# Imports and environment check — run this cell first.
import os

from dotenv import load_dotenv

# Load .env from this folder. The Tavily tool and the model client both
# read their keys straight from the environment.
load_dotenv()

required = ["OPENWEATHER_API_KEY", "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "TAVILY_API_KEY"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise SystemExit(f"Missing keys in .env: {', '.join(missing)} — see .env.example.")

print("API keys loaded ✔")


## 2. Shared configuration

Everything configurable — keys, endpoints, model settings and the agent
prompt — lives in `constants.py`, so this notebook stays focused on wiring
the agent. `constants.py` calls `load_dotenv()` itself, so importing it
already picks up your `.env`.


In [ ]:
# Shared config: keys, OpenWeather endpoint, model settings, prompt.
import sys
from pathlib import Path

# constants.py lives in src/ — make it importable from the notebook.
sys.path.insert(0, str(Path("src").resolve()))

from constants import (
    LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, MODEL_REASONING_EFFORT,
    MODEL_TEMPERATURE, OPENWEATHER_API_KEY, OPENWEATHER_URL,
    PROMPT, REQUEST_TIMEOUT_SECONDS,
)

print(f"Model: {LLM_MODEL}")
print(f"Base URL: {LLM_BASE_URL}")
print(f"temperature={MODEL_TEMPERATURE}, reasoning_effort={MODEL_REASONING_EFFORT}")


## 3. The tool: `get_weather(city)`

A custom Strands tool. The `@tool` decorator exposes the function to the
agent; its name, signature and docstring become the tool spec the model
sees. Built-in tools (`tavily`, `calculator`) come from `strands_tools` and
are registered the same way.

Error handling: unknown city (`404`) returns a readable message instead of
raising — the agent can react to it rather than crash.


In [ ]:
import requests

from constants import OPENWEATHER_API_KEY, OPENWEATHER_URL, REQUEST_TIMEOUT_SECONDS
from strands import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city using OpenWeather.

    Args:
        city: City name, e.g. "Kathmandu".

    Returns:
        Human-readable summary of current temperature and conditions.
    """
    resp = requests.get(
        OPENWEATHER_URL,
        params={"q": city, "units": "metric", "appid": OPENWEATHER_API_KEY},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    if resp.status_code == 404:
        return f"Weather not found for city '{city}'."
    resp.raise_for_status()
    data = resp.json()
    temp = data["main"]["temp"]
    feels = data["main"]["feels_like"]
    desc = data["weather"][0]["description"]
    name = data["name"]
    return (
        f"Current weather in {name}: {temp:.1f}C (feels like {feels:.1f}C), {desc}."
    )

# Try it directly, bypassing the agent — expect something like:
# 'Current weather in Kathmandu: 22.4C (feels like 22.1C), scattered clouds.'
get_weather("Kathmandu")


## 4. Wire up the agent

- `OpenAIModel` points Strands at any OpenAI-compatible endpoint — here the
  base URL + key come from `.env` (e.g. OpenRouter).
- `Agent(model=..., tools=[...])` attaches the three tools. From this point,
  Strands' agent loop drives everything: tool calls, result round-trips,
  and the final answer.


In [ ]:
from strands import Agent
from strands.models.openai import OpenAIModel
from strands_tools import calculator, tavily

MODEL = OpenAIModel(
    client_args={"api_key": LLM_API_KEY, "base_url": LLM_BASE_URL},
    model_id=LLM_MODEL,
    params={
        "temperature": MODEL_TEMPERATURE,
        "reasoning_effort": MODEL_REASONING_EFFORT,
    },
)

agent = Agent(model=MODEL, tools=[get_weather, tavily, calculator])
print("Agent ready ✔ (tools: get_weather, tavily, calculator)")


## 5. Run the agent

Set `CITY` to whatever you want to visit and run the cell. The prompt (from
`constants.py`) tells the model to call the tools in order: weather →
attraction search → cost calculation → write the itinerary.

**Expected output** (varies with live weather and search results): the
tool-call log lines from Strands, followed by the final itinerary as shown
at the top of this notebook. A typical final answer:

```
One-Day Trip to Kathmandu

Weather: Current weather in Kathmandu: 22.4C (feels like 22.1C),
scattered clouds.

Morning: Kathmandu Durbar Square ...
Afternoon: Swayambhunath Stupa (Monkey Temple) ...
Evening: Boudhanath Stupa ...

Cost: entry fees $12 + transport $8 + meals $15 = $35 per person
```


In [ ]:
CITY = "Kathmandu"  # change me — any city name works

print(agent(PROMPT.format(city=CITY)))


## 6. Recap

1. **Config** — keys, endpoints, model settings and the prompt come from `constants.py`.
2. **Tools** — one custom tool (`get_weather`, OpenWeather) plus two built-ins (`tavily` search, `calculator`).
3. **Agent** — Strands' automatic loop handles tool round-trips; the prompt pins the step order.
4. **Output** — a one-day itinerary with a weather line, three attractions, and a costed schedule.

Try changing `CITY` above, or edit `PROMPT` in `constants.py` to ask for a
weekend (multi-day) plan instead.
